# Week 4 — Task 1: Earnings Data Feasibility Check

**Objective**: Determine whether yfinance can supply enough earnings-date + EPS-surprise records for Option A (earnings announcements). If the aggregate usable-event ratio is ≥ 50%, adopt Option A; otherwise fall back to Option B (FOMC meeting dates).

**Decision rule**:
- Usable event = one earnings announcement for which yfinance returns a non-null EPS Estimate, Reported EPS, *and* Surprise(%) within the 2020-08-30 → 2025-08-30 window
- Theoretical maximum = 50 tickers × 20 calendar quarters = 1,000 events
- Threshold: ≥ 50% → Option A; < 50% → Option B

In [1]:
import sys
from pathlib import Path

repo_root = Path("../..").resolve()
sys.path.insert(0, str(repo_root / "src"))

import yfinance as yf
import pandas as pd
import numpy as np
import time
from universe import SP500_TOP50

print(f"Universe: {len(SP500_TOP50)} non-SPY tickers")
print(f"Study window: 2020-08-30 → 2025-08-30")
print(f"Theoretical max events: {len(SP500_TOP50)} × 20 = {len(SP500_TOP50) * 20}")

Universe: 50 non-SPY tickers
Study window: 2020-08-30 → 2025-08-30
Theoretical max events: 50 × 20 = 1000


## Step 1: Probe the yfinance API

Verify the method name, output schema, and date range available before running the full pull.

In [2]:
# Probe with AAPL to confirm method name and output schema
t = yf.Ticker("AAPL")
sample = t.get_earnings_dates(limit=12)

print("Method: Ticker.get_earnings_dates(limit=N)")
print(f"Return type: {type(sample)}")
print(f"Columns: {sample.columns.tolist()}")
print(f"Index name: {sample.index.name}")
print(f"Index dtype: {sample.index.dtype}")
print()
print("Sample output (12 rows, newest first):")
sample

Method: Ticker.get_earnings_dates(limit=N)
Return type: <class 'pandas.DataFrame'>
Columns: ['EPS Estimate', 'Reported EPS', 'Surprise(%)']
Index name: Earnings Date
Index dtype: datetime64[us, America/New_York]

Sample output (12 rows, newest first):


,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2026-10-29 16:00:00-04:00,1.98,NaN,NaN
2026-07-30 16:00:00-04:00,1.89,2.02,6.74
2026-04-30 16:00:00-04:00,1.94,2.01,3.46
2026-01-29 16:00:00-05:00,2.67,2.84,6.25
2025-10-30 16:00:00-04:00,1.77,1.85,4.52
2025-07-31 16:00:00-04:00,1.43,1.57,9.48
2025-05-01 16:00:00-04:00,1.62,1.65,1.69
2025-01-30 16:00:00-05:00,2.35,2.40,2.26
2024-10-31 16:00:00-04:00,0.95,0.97,2.48


Also probe a BMO-classified ticker (JPM) to confirm the timestamp format for before-market-open announcements:

In [3]:
# Probe a BMO ticker (JPM) to show the announcement-hour format for before-market-open events
t_jpm = yf.Ticker("JPM")
sample_jpm = t_jpm.get_earnings_dates(limit=12)

print("JPM get_earnings_dates(limit=12):")
sample_jpm

JPM get_earnings_dates(limit=12):


,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2026-10-13 08:00:00-04:00,5.89,NaN,NaN
2026-07-14 06:00:00-04:00,5.80,6.14,5.86
2026-04-14 06:00:00-04:00,5.51,5.94,7.78
2026-01-13 06:00:00-05:00,4.82,4.63,-3.91
2025-10-14 06:00:00-04:00,4.87,5.07,4.01
2025-07-15 06:00:00-04:00,4.48,4.96,10.62
2025-04-11 06:00:00-04:00,4.64,5.07,9.15
2025-01-15 06:00:00-05:00,4.04,4.81,19.18
2024-10-11 06:00:00-04:00,3.99,4.37,9.63


## Step 2: Full Feasibility Check — All 50 Tickers

For each ticker:
1. Pull `get_earnings_dates(limit=50)` — enough to cover 12+ years; our window needs ~20 quarters
2. Filter to 2020-08-30 → 2025-08-30
3. Count events where **all three** of `EPS Estimate`, `Reported EPS`, `Surprise(%)` are non-null
4. Classify announcement timing (BMO / AMC / mixed) using the timestamp hour in US/Eastern time

In [4]:
START = pd.Timestamp("2020-08-30", tz="UTC")
END   = pd.Timestamp("2025-08-30", tz="UTC")

rows = []
for ticker in SP500_TOP50:
    try:
        t = yf.Ticker(ticker)
        ed = t.get_earnings_dates(limit=50)
        if ed is None or len(ed) == 0:
            rows.append({
                "ticker": ticker, "total_in_window": 0,
                "usable": 0, "announcement_timing": "none"
            })
            time.sleep(0.2)
            continue

        # Normalize to UTC for window filter
        ed.index = ed.index.tz_convert("UTC")
        mask = (ed.index >= START) & (ed.index <= END)
        window = ed[mask]

        # Timing classification using US/Eastern hours
        et_hours = window.index.tz_convert("US/Eastern").hour
        if len(et_hours) == 0:
            timing = "none"
        elif all(h <= 10 for h in et_hours):
            timing = "BMO"
        elif all(h >= 14 for h in et_hours):
            timing = "AMC"
        else:
            timing = "mixed"

        total  = len(window)
        usable = int(
            (window["EPS Estimate"].notna()
             & window["Reported EPS"].notna()
             & window["Surprise(%)"].notna()).sum()
        )
        rows.append({
            "ticker": ticker,
            "total_in_window": total,
            "usable": usable,
            "announcement_timing": timing
        })
    except Exception as exc:
        rows.append({
            "ticker": ticker, "total_in_window": 0,
            "usable": 0, "announcement_timing": f"error: {exc}"
        })
    time.sleep(0.25)

coverage = pd.DataFrame(rows)
print(f"Tickers processed: {len(coverage)}")
coverage

Tickers processed: 50


,ticker,total_in_window,usable,announcement_timing
0,NVDA,20,20,AMC
1,AAPL,20,20,AMC
2,MSFT,20,20,AMC
3,AMZN,20,20,mixed
4,META,20,20,AMC
5,GOOGL,20,20,AMC
6,TSLA,20,20,AMC
7,AVGO,20,20,mixed
8,BRK-B,20,20,BMO
9,LLY,20,20,BMO


## Step 3: Aggregate Coverage and Decision

In [5]:
THEORETICAL_MAX = 50 * 20
total_usable   = int(coverage["usable"].sum())
usable_ratio   = total_usable / THEORETICAL_MAX * 100
THRESHOLD_PCT  = 50.0

print("=" * 55)
print(f"  Theoretical maximum  : {THEORETICAL_MAX:>6}")
print(f"  Total usable events  : {total_usable:>6}")
print(f"  Aggregate ratio      : {usable_ratio:.4f}%")
print(f"  Threshold            : {THRESHOLD_PCT:.1f}%")
print("=" * 55)

if usable_ratio >= THRESHOLD_PCT:
    decision = "Option A — Earnings Announcements"
    reason   = f"Coverage {usable_ratio:.4f}% ≥ {THRESHOLD_PCT:.0f}% threshold"
else:
    decision = "Option B — FOMC Meeting Dates"
    reason   = f"Coverage {usable_ratio:.4f}% < {THRESHOLD_PCT:.0f}% threshold (fallback)"

print(f"\nDECISION: {decision}")
print(f"REASON  : {reason}")

# Timing breakdown
print("\nAnnouncement-timing breakdown:")
print(coverage["announcement_timing"].value_counts().to_string())

  Theoretical maximum  :   1000
  Total usable events  :    999
  Aggregate ratio      : 99.9000%
  Threshold            : 50.0%

DECISION: Option A — Earnings Announcements
REASON  : Coverage 99.9000% ≥ 50% threshold

Announcement-timing breakdown:
announcement_timing
BMO      21
AMC      17
mixed    12


## Step 4: Notes on Coverage Anomalies

**CRM — 19 events instead of 20**: Salesforce uses a January 31 fiscal year-end (Q1 = Feb–Apr, Q2 = May–Jul, Q3 = Aug–Oct, Q4 = Nov–Jan). Its Q2 FY2021 earnings (covering the quarter ending July 31, 2020) were released on 2020-08-25, five days before our data-window start of 2020-08-30. The event record exists in yfinance but falls outside the window; the 19 usable CRM events represent all events that can in principle be studied.

**Announcement timing for "mixed" tickers**: "mixed" is a per-ticker summary label, not a per-event data gap. Every individual earnings event carries an exact timestamp down to the hour — each event's own announcement hour is what Step 2's classification logic reads directly (e.g. `16:00:00-04:00` for AAPL/AMC as shown in the Step 1 probe above, and `06:00:00-04:00` for JPM/BMO as shown in the JPM probe above). A ticker is classified "mixed" only when its announcement timing varies across quarters, not because any single event's timing is unknown. Task 2's `align_to_trading_days()` function will use each event's own announcement hour to assign t=0 per event: same trading day for BMO (hour ≤ 10 ET), next trading day for AMC (hour ≥ 14 ET). No per-event ambiguity carries into the event panel.

## Quality Checks

In [6]:
checks = {}

# 1. All 50 tickers processed
checks["50 tickers processed"] = len(coverage) == 50

# 2. Total usable > 0
checks["Total usable > 0"] = total_usable > 0

# 3. Ratio is exact (not estimated)
checks["Exact ratio computed (no rounding)"] = isinstance(usable_ratio, float) and usable_ratio == total_usable / THEORETICAL_MAX * 100

# 4. No tickers with 0 usable events (all data present)
checks["No tickers with 0 usable events"] = (coverage["usable"] == 0).sum() == 0

# 5. Decision applied mechanically from decision rule
checks[f"Decision applied from rule (>={THRESHOLD_PCT}% → Option A)"] = (
    (usable_ratio >= THRESHOLD_PCT and decision.startswith("Option A")) or
    (usable_ratio < THRESHOLD_PCT and decision.startswith("Option B"))
)

# 6. Usable ≤ total_in_window for each ticker
checks["usable <= total_in_window for all tickers"] = (coverage["usable"] <= coverage["total_in_window"]).all()

for desc, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {desc}")

assert all(checks.values()), "One or more quality checks failed — investigate before proceeding to Task 2"
print(f"\nAll {len(checks)} quality checks passed.")

[PASS] 50 tickers processed
[PASS] Total usable > 0
[PASS] Exact ratio computed (no rounding)
[PASS] No tickers with 0 usable events
[PASS] Decision applied from rule (>=50.0% → Option A)
[PASS] usable <= total_in_window for all tickers

All 6 quality checks passed.
